In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("../data/backtest_net_returns.csv", index_col=0, parse_dates=True)
rf = data.pop("rf")
excess = data.sub(rf, axis=0)

def sharpe(x):
    return x.mean(axis=0) / x.std(axis=0, ddof=1) * np.sqrt(12)

def stationary_bootstrap_indices(n, mean_block, rng):
    """Politis-Romano stationary bootstrap: blocks of random length, average `mean_block`."""
    idx = np.empty(n, dtype=int)
    idx[0] = rng.integers(n)
    for i in range(1, n):
        if rng.random() < 1 / mean_block:
            idx[i] = rng.integers(n)          # start a new block
        else:
            idx[i] = (idx[i - 1] + 1) % n     # continue the current block
    return idx

N_BOOT = 5000
MEAN_BLOCK = 6
rng = np.random.default_rng(42)

X = excess.values
n = len(X)
boot = np.empty((N_BOOT, X.shape[1]))
for b in range(N_BOOT):
    sample = X[stationary_bootstrap_indices(n, MEAN_BLOCK, rng)]
    boot[b] = sample.mean(axis=0) / sample.std(axis=0, ddof=1) * np.sqrt(12)

boot_sharpes = pd.DataFrame(boot, columns=excess.columns)

pd.DataFrame({
    "sharpe": sharpe(excess),
    "ci_low": boot_sharpes.quantile(0.025),
    "ci_high": boot_sharpes.quantile(0.975),
}).round(3)

,sharpe,ci_low,ci_high
equal_weight,0.472,0.046,0.967
naive_mv,0.401,-0.075,0.958
ledoit_wolf_mv,0.397,-0.074,0.943
resampled_mv,0.586,0.132,1.120
bl_no_views,0.605,0.161,1.112
bl_momentum,0.595,0.100,1.116


In [2]:
def diff_table(benchmark):
    diffs = boot_sharpes.sub(boot_sharpes[benchmark], axis=0).drop(columns=benchmark)
    point = (sharpe(excess) - sharpe(excess)[benchmark]).drop(benchmark)
    return pd.DataFrame({
        "sharpe_diff": point,
        "ci_low": diffs.quantile(0.025),
        "ci_high": diffs.quantile(0.975),
        "share_better": (diffs > 0).mean(),
    }).round(3)

print("Versus naive mean-variance:")
print(diff_table("naive_mv"))
print("\nVersus equal weight:")
print(diff_table("equal_weight"))

Versus naive mean-variance:
                sharpe_diff  ci_low  ci_high  share_better
equal_weight          0.071  -0.309    0.440         0.627
ledoit_wolf_mv       -0.004  -0.025    0.013         0.336
resampled_mv          0.184   0.008    0.348         0.979
bl_no_views           0.203  -0.162    0.615         0.845
bl_momentum           0.194  -0.282    0.659         0.778

Versus equal weight:
                sharpe_diff  ci_low  ci_high  share_better
naive_mv             -0.071  -0.440    0.309         0.373
ledoit_wolf_mv       -0.075  -0.444    0.305         0.367
resampled_mv          0.113  -0.176    0.410         0.780
bl_no_views           0.132   0.028    0.257         0.993
bl_momentum           0.123  -0.188    0.420         0.786


In [3]:
def bootstrap_sharpes(X, mean_block, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(X)
    out = np.empty((n_boot, X.shape[1]))
    for b in range(n_boot):
        sample = X[stationary_bootstrap_indices(n, mean_block, rng)]
        out[b] = sample.mean(axis=0) / sample.std(axis=0, ddof=1) * np.sqrt(12)
    return pd.DataFrame(out, columns=excess.columns)

rows = []
for block in [1, 3, 6, 12]:
    bs = bootstrap_sharpes(excess.values, block)
    for strat, bench in [("resampled_mv", "naive_mv"), ("bl_no_views", "equal_weight")]:
        d = bs[strat] - bs[bench]
        rows.append({
            "mean_block": block,
            "comparison": f"{strat} vs {bench}",
            "ci_low": d.quantile(0.025),
            "ci_high": d.quantile(0.975),
            "share_better": (d > 0).mean(),
        })

pd.DataFrame(rows).round(3)

,mean_block,comparison,ci_low,ci_high,share_better
0,1,resampled_mv vs naive_mv,0.018,0.359,0.986
1,1,bl_no_views vs equal_weight,0.029,0.243,0.994
2,3,resampled_mv vs naive_mv,0.015,0.351,0.984
3,3,bl_no_views vs equal_weight,0.028,0.248,0.992
4,6,resampled_mv vs naive_mv,0.008,0.348,0.979
5,6,bl_no_views vs equal_weight,0.028,0.257,0.993
6,12,resampled_mv vs naive_mv,0.000,0.361,0.975
7,12,bl_no_views vs equal_weight,0.030,0.263,0.997
